# Notebook 3 — Model Training

This notebook trains two deep-learning architectures on the FER2013 dataset
and saves the resulting models and training histories to disk.

- **Model A — Custom CNN**: a four-block convolutional network trained from
  scratch on 48 × 48 grayscale images. Serves as the interpretable baseline.
- **Model B — MobileNetV2 fine-tuned**: a lightweight pretrained backbone
  adapted to FER2013 in two stages. Expected to generalise better due to
  ImageNet pretraining.

All evaluation (accuracy curves, confusion matrices, per-class metrics) is
deferred to Notebook 4. Here we only run a brief smoke test after each model
is saved to confirm the output pipeline works end-to-end.

## Section 1 — Setup

We fix all random seeds before any other operation to ensure reproducibility
across runs. Seeds are set for Python's `random` module, NumPy, and TensorFlow
independently because each maintains its own PRNG state. We also verify GPU
availability here; training without a GPU is possible but wall-clock time
increases roughly 10×.

In [ ]:
import json
import random
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.models.cnn_custom import build_cnn_custom
from src.models.mobilenet_finetune import build_mobilenet_head, unfreeze_top_layers
from src.models.trainer import get_default_callbacks, save_history, train_model

SEED = config.RANDOM_SEED
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU available: {gpus}")
else:
    print("WARNING: No GPU detected. Training will be significantly slower.")

## Section 2 — Load preprocessed data

We reload the `.npz` files and class-weights JSON produced by Notebook 2
rather than re-running the preprocessing pipeline. This ensures that the
exact same normalised tensors and train/val/test split are used for both
models, and reduces notebook startup time from several minutes to a few
seconds. JSON dictionary keys are strings by default; we cast them to `int`
because the Keras `class_weight` argument requires integer keys.

In [ ]:
gray = np.load(config.PROCESSED_DIR / "fer2013_gray.npz")
rgb  = np.load(config.PROCESSED_DIR / "fer2013_rgb64.npz")

with open(config.PROCESSED_DIR / "class_weights.json") as f:
    class_weights_raw = json.load(f)
class_weights = {int(k): v for k, v in class_weights_raw.items()}

X_train_gray, y_train = gray["X_train"], gray["y_train"]
X_val_gray,   y_val   = gray["X_val"],   gray["y_val"]
X_test_gray,  y_test  = gray["X_test"],  gray["y_test"]

X_train_rgb = rgb["X_train"]
X_val_rgb   = rgb["X_val"]
X_test_rgb  = rgb["X_test"]

print(f"Grayscale  — train: {X_train_gray.shape}  val: {X_val_gray.shape}  test: {X_test_gray.shape}")
print(f"RGB 64×64  — train: {X_train_rgb.shape}  val: {X_val_rgb.shape}  test: {X_test_rgb.shape}")
print(f"Labels     — y_train: {y_train.shape}  y_val: {y_val.shape}  y_test: {y_test.shape}")
print(f"Class weights: {class_weights}")

## Section 3 — Custom CNN (Model A)

### Section 3.1 — Build the architecture

The custom CNN uses four convolutional blocks with doubling filter counts
(32 → 64 → 128 → 256), each followed by BatchNorm, ReLU, MaxPooling, and
Dropout(0.25). Two dense layers (512, 256) with Dropout(0.5) precede the
7-class softmax head. This depth balances representational capacity against
overfitting risk on 48 × 48 images — a fifth block would multiply parameters
with minimal feature-resolution gain given the small spatial size.

In [ ]:
model_cnn = build_cnn_custom(input_shape=(48, 48, 1), num_classes=7)
model_cnn.summary()

### Section 3.2 — Configure training callbacks

Three standard callbacks are attached. `EarlyStopping` (patience=7) halts
training when validation loss stagnates and restores the best weights seen
during the run. `ReduceLROnPlateau` (patience=3, factor=0.5) halves the
learning rate after three epochs without improvement, letting the optimiser
converge into narrow loss minima that a large fixed LR would overshoot.
`ModelCheckpoint` writes the best checkpoint independently, so it is
recoverable even if the process is interrupted.

In [ ]:
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
config.HISTORIES_DIR.mkdir(parents=True, exist_ok=True)

callbacks_cnn = get_default_callbacks("cnn_custom", config.MODELS_DIR)
print("Callbacks configured:")
for cb in callbacks_cnn:
    print(f"  {cb.__class__.__name__}")